## IO database disaggregation with MARIO

This notebook follows the workflow in the slides: load GTAP, apply the model geography, prepare the new-sector inputs, run the disaggregation and explore the saved result.

[MARIO](https://www.marioproject.com/en/latest/) manages the IO database. The sectors data are pre-processed in the [ENTICE database disaggregation repository](https://github.com/SESAM-Polimi/Database-disaggregation); [CVXLab](https://cvxlab.readthedocs.io/en/latest/) builds and solves the optimisation problem called by MARIO.

### Before starting
Use a Python environment with CVXLab 1.1.0 and the repository dependencies installed. Before starting the notebook, switch each repository to the required branch:

- [MARIO](https://github.com/it-is-me-mario/MARIO): `codex/mrio-memory`.
- [Database disaggregation](https://github.com/SESAM-Polimi/Database-disaggregation): `codex/notebook-api`. This branch is currently local and must be published before other users can clone it from GitHub.

In `settings.json`, set `data_root` to your local SharePoint **Data split** folder, `mario_repository` to the MARIO checkout, and `disaggregation_repository` to the Database disaggregation checkout. Dataset paths are relative to `data_root`. Restart the kernel if you change either repository location.

In `settings.json`, `database_format` selects `parquet` or `csv`; `database_path` points to the corresponding folder relative to `data_root`. The workshop uses the existing native Parquet database. For the original CSVs, select `csv` and point to the extracted GTAP bundle.

`geography` selects the shared region mapping, `workbook` selects the prepared full input plan, and `result_database` selects the published run. Keep these selections together when choosing another example. Restart the kernel before running this notebook with the updated MARIO checkout.

The published example is a partial disaggregation: unsuccessful activities remain in their parents. The notebook prints the actual sector counts. The optional preparation and optimisation cells are commented out for the workshop. Running the optimisation requires Gurobi and its licence.


In [ ]:
from pathlib import Path
import json
import os
import sys
import tempfile

cfg = json.loads(Path("settings.json").read_text())
DATA_ROOT = Path(cfg["data_root"]).expanduser()
os.environ["ENTICE_DATA_ROOT"] = str(DATA_ROOT)
sys.path[:0] = [cfg["mario_repository"], str(Path(cfg["disaggregation_repository"]) / "src")]

import mario
from IPython.display import FileLink, display
from database_disaggregation import Geography, SplitPlan, prepare_model_inputs, analysis, reporting
from database_disaggregation.core.residual_labels import load_checkpoint_with_codes

GTAP = DATA_ROOT / cfg["database_path"]
RUN = DATA_ROOT / cfg["result_database"]
WORK = Path(tempfile.mkdtemp(prefix="mario-disaggregation-"))
inventory_file = DATA_ROOT / cfg["workbook"]
plan = SplitPlan.read(inventory_file)
result = None
saved_report = None


---
## 1. Load and aggregate GTAP
### Load GTAP Power MRIO for 2023

MARIO reads the native Parquet database with `parse_from_parquet`, or the original CSV tables with `parse_gtap`, according to `settings.json`. Both return a MARIO `Database`. Its main monetary blocks are intermediate transactions (`Z`), final demand (`Y`) and value added (`V`); extensions are stored separately.

The parsed database stays in memory as `native_db`. We use it both to create the aggregated database and to prepare the new sectors’ data.

References: [Parquet reader](https://www.marioproject.com/en/latest/api_document/mario.parse_from_parquet.html) · [GTAP reader](https://www.marioproject.com/en/latest/api_document/mario.parse_gtap.html)


In [ ]:
if cfg["database_format"] == "parquet":
    native_db = mario.parse_from_parquet(str(GTAP), table="IOT", mode="flows", year=2023)
elif cfg["database_format"] == "csv":
    native_db = mario.parse_gtap(path=str(GTAP), input_format="csv", year=2023)
else:
    raise ValueError("database_format must be 'parquet' or 'csv'")


### Apply the model geography

The shared mapping assigns each native GTAP region to one model region. MARIO sums the corresponding rows and columns, keeping the sector classification unchanged. The selected partition can be built entirely by aggregating native GTAP regions.

The same mapping is passed to data preparation. Production, trade and regional cost structures therefore use the same geography as the database. The native database remains available for reconciliation weights.

Reference: [Database.aggregate](https://www.marioproject.com/en/latest/api_document/mario.Database.aggregate.html)


In [ ]:
geography = Geography.read(DATA_ROOT / cfg["geography"])
db = native_db.aggregate(
    None, levels="Region", region_aggregation=geography.mapping,
    zero_output_epsilon=None, inplace=False,
)
print(f"{len(db.get_index('Region'))} regions, {len(db.get_index('Sector'))} sectors")


---
## 2. Prepare the disaggregation data

The repository combines the collected production, trade and cost-structure data with the native sector inventories. It applies the existing classification, unit and reconciliation rules and prepares a common MARIO input workbook for the selected geography.

```text
Collected sector data + native MARIO database
                     ↓
       Database disaggregation repository
        source rules and regional mapping
                     ↓
              add_sectors.xlsx
```

The extended project collection includes barley (`BAR`), steel via BF–BOF with natural-gas injection (`SGI`) and revised coal products (`PCP`). `prepare_model_inputs` reuses the preparation implemented for this run. It receives `native_db` directly, uses its native regional weights and writes a new input folder. The repository's existing path configuration selects the collection and native-inventory folders.

For the workshop, the full prepared workbook is already selected in `settings.json`. The cell below is commented out; uncomment it to rebuild the inputs before a fresh solve. Source data and the published result are preserved.


In [ ]:
# prepared = prepare_model_inputs(
#     native_db, WORK / "prepared_inputs", geography=geography, steps=plan.steps,
# )
# inventory_file = prepared / "add_sectors.xlsx"


### Read the prepared workbook

`Master` defines the planned new sectors and their parents. Inventory sheets contain regional input recipes; `Total outputs` and `Trades` provide the monetary targets. Cluster sheets define grouped sectors, regions and factors. `Intermediate steps` contains the execution order, with one sector in each group for this example.

Open the selected `add_sectors.xlsx` in Excel to inspect these sheets. The workbook retains the full plan even when some sectors could not be added to the saved result. MARIO reads this standard format directly.


In [ ]:
db.read_add_sectors_excel(inventory_file, read_inventories=True, split=True)


---
## 3. Run the optimization model

MARIO combines the aggregated database and prepared inputs, builds the split through CVXLab and calls the selected solver. `inplace=False` returns the disaggregated database as `result`, keeping the initial database available.

The selected run used Gurobi with the quadratic model and `on_step_failure="continue"`. Each group is attempted once. If a recognised optimization failure occurs, that sector remains in its parent and later groups are attempted. Invalid inputs and programming errors still stop execution. Solver settings and model options are recorded in `settings.json`; checkpoints from a fresh solve go to the temporary folder.

The cell below is commented out for the workshop. Uncomment it after preparing the inputs to run a fresh reconstruction. Environmental extensions (`E`) require recalculation after this monetary split.


In [ ]:
# result = db.add_sectors(
#     "inventories", split=True, inplace=False,
#     intermediate_steps=True, save_intermediates=WORK / "checkpoints",
#     on_step_failure=cfg["on_step_failure"],
#     solver=cfg["solver"], solver_parameters=cfg["solver_parameters"],
#     split_model_options=cfg["split_model_options"],
# )


### Load the saved result

If the optimization cell is skipped, this loads the published result and selects its matching numerical report. The repository loader verifies the checkpoint, preserves its independently constrained output `X`, and restores stable sector codes from the published residual names. The dashboard still shows the published names and parent information.

The count below distinguishes sectors actually added from those planned. Uncompleted activities remain inside their parents. The published run has numerical residuals recorded for review; loading it does not imply that every strict numerical check passed or that economic validation is complete.

If a fresh solution has been computed instead, it remains `result` and the saved report is not attached to it. The historical native Parquet database and a fresh parse of the original CSVs have not yet been verified as numerically equivalent.


In [ ]:
if result is None:
    result = load_checkpoint_with_codes(RUN)
    saved_report = RUN / "benchmarking.json"

added = set(plan.sectors) & set(result.get_index("Sector"))
print(f"{len(result.get_index('Region'))} regions · {len(result.get_index('Sector'))} sectors")
print(f"{len(added)} / {len(plan.sectors)} planned new sectors present")


---
## 4. Explore the results

The dashboard shows sector status and parent, production, bilateral trade and the distribution of a selected sector’s output across users. The heatmap sums all supplying origins and normalises each destination column to 100% across using sectors and **all native final-demand categories**, including households, government, investment and other categories present in the database. All negative use shares remain visible in red.

For the saved run, the report supplies the recorded target checks, numerical audit and compatible cross-database comparisons for this exact database and geography. Activities left in their parents and comparisons outside strict tolerances remain visible. Target checks measure agreement with prepared model inputs; they are not independent validation. `E` and environmental footprints remain pending.

For a fresh reconstruction, the cell computes target checks for the sectors actually present and generates the current result views. External comparisons and a full numerical audit can be added after benchmarking that new run; an earlier run’s audit is not reused.

The cell writes one standalone HTML dashboard that can be opened beside the slides.


In [ ]:
dashboard = Path("results/MARIO_workshop_dashboard.html")
if saved_report is not None:
    reporting.write_dashboard(result, dashboard, benchmark=saved_report)
else:
    checks = analysis.check_targets(result, inventory_file, sectors=result.get_index("Sector"))
    reporting.write_dashboard(result, dashboard, checks=checks, workbook=inventory_file)
display(FileLink(dashboard))
